# Examples

In [1]:
from DRlm import DRlm
from drol import DRoL
import numpy as np
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings("ignore")

## Example 1: DRlm - Regression

### Data Generating Process

In [2]:
np.random.seed(0)  # For reproducibility
# number of groups
L = 2
# dimension
p = 100

# mean vector for source
mean_source = np.zeros(p)

# covariance matrix for source
def A1gen(rho, p):
    A1 = np.zeros((p, p))
    for i in range(p):
        for j in range(p):
            A1[i, j] = rho ** abs(i - j)
    return A1

cov_source = A1gen(0.6, p)

# 1st group's source data
n1 = 100
X1 = multivariate_normal.rvs(mean=mean_source, cov=cov_source, size=n1)
b1 = np.zeros(p)
b1[0:5] = np.arange(1, 6) / 20
b1[97:100] = [0.5, -0.5, -0.5]
Y1 = X1 @ b1 + np.random.normal(size=n1)

# 2nd group's source data
n2 = 100
X2 = multivariate_normal.rvs(mean=mean_source, cov=cov_source, size=n2)
b2 = np.zeros(p)
b2[5:10] = np.arange(1, 6) / 20
b2[97:100] = 0.5 * np.array([0.5, -0.5, -0.5])
Y2 = X2 @ b2 + np.random.normal(size=n2)

# Target Data, covariate shift
n0 = 100
mean0 = np.zeros(p) + 0.1
cov0 = cov_source.copy()

# diagonal elements
for i in range(p):
    cov0[i, i] = 1.5

# first 5x5 block off-diagonal
for i in range(5):
    for j in range(5):
        if i != j:
            cov0[i, j] = 0.9

# last 2x2 block off-diagonal (indices 98~100 in R = 97~99 in Python)
for i in range(98, 100):
    for j in range(98, 100):
        if i != j:
            cov0[i, j] = 0.9

X0 = multivariate_normal.rvs(mean=mean0, cov=cov0, size=n0)

Xlist = [X1, X2]
ylist = [Y1, Y2]


In [3]:
# dimension p=100
loading_mat = np.zeros((100, 2))
loading_mat[95:100, 0] = 0.4  
loading_mat[98:100, 1] = 0.8

loading_mat = loading_mat.T


### Implementation

In [4]:
reg = DRlm.Regression(f_learner='high_d', verbose=True)
reg.fit(Xlist, ylist, loading_mat, X0=X0)
reg.infer(M=200, alpha=0.05, alpha_thres=0.01)

## time cost: 6.6s

Argument 'loading_intercept' set to False because intercept is False
start fitting-----
======> Bias Correction for initial estimators....
---> Computing for loading (1/2)...
The projection direction is identified at xi = 0.026710 at step = 6.0
---> Computing for loading (2/2)...
The projection direction is identified at xi = 0.040065 at step = 5.0
---> Computing for loading (1/2)...
The projection direction is identified at xi = 0.026710 at step = 6.0
---> Computing for loading (2/2)...
The projection direction is identified at xi = 0.026710 at step = 6.0
======> Bias Correction for matrix Gamma....
---> Computing for loading (1/1)...
The projection direction is identified at xi = 0.026710 at step = 6.0
---> Computing for loading (1/1)...
The projection direction is identified at xi = 0.026710 at step = 6.0
---> Computing for loading (1/1)...
The projection direction is identified at xi = 0.026710 at step = 6.0
---> Computing for loading (1/1)...
The projection direction is identified

### Results

In [5]:
reg.summary()

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.1485   0.8515

Fitted Plug-in Estimations (Maximin Effects):

index     |        1        2        3        4        5        6        7        8        9       10
coef_     |   0.0074  -0.0132  -0.0762  -0.0011   0.0758   0.1428   0.0008   0.2126   0.0211   0.1099
index     |       11       12       13       14       15       16       17       18       19       20
coef_     |   0.0000   0.0015   0.0000  -0.0293   0.0014   0.0000   0.0365   0.0000   0.0064   0.0000
index     |       21       22       23       24       25       26       27       28       29       30
coef_     |   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000  -0.0049   0.0673
index     |       31       32       33       34       35       36       37       38       39       40
coef_     |   0.0203   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0003   0.0000
index     |       41       42       43      

In [6]:
reg.summary(dim_search=[2])

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.1485   0.8515

Fitted Plug-in Estimations (Maximin Effects):

index     |        1        2        3        4        5        6        7        8        9       10
coef_     |   0.0074  -0.0132  -0.0762  -0.0011   0.0758   0.1428   0.0008   0.2126   0.0211   0.1099
index     |       11       12       13       14       15       16       17       18       19       20
coef_     |   0.0000   0.0015   0.0000  -0.0293   0.0014   0.0000   0.0365   0.0000   0.0064   0.0000
index     |       21       22       23       24       25       26       27       28       29       30
coef_     |   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000  -0.0049   0.0673
index     |       31       32       33       34       35       36       37       38       39       40
coef_     |   0.0203   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0003   0.0000
index     |       41       42       43      

In [7]:
reg.predict()

{'pred': array([ 0.59980916,  0.0349804 , -0.13899486,  0.13319991, -0.68127574,
        -0.48184502, -0.17022069,  0.15677349, -0.77144535,  1.13837822,
         0.46048242,  0.57196422, -0.43519576, -0.55605458,  0.2963047 ,
        -0.16059679,  0.21675115, -0.36824958, -0.83239964, -0.49934856,
        -0.25562494,  0.42434761,  0.91744696,  0.23179753, -0.58859777,
         0.8151657 ,  0.06801001,  0.0849275 ,  0.83306642,  0.04003204,
        -0.03601859, -0.2058141 ,  0.34361413, -0.02040277, -0.85262232,
        -0.19408246, -0.2646345 , -0.55323176, -0.3640268 , -0.22893331,
         0.03681625, -0.67315086,  0.6576169 ,  1.21628339,  0.01113962,
         0.02714514, -0.71359756, -0.15495462,  0.47990769,  0.26907842,
         0.24443202,  0.09720009,  0.52310755,  0.05343414, -0.98189137,
        -1.00192819,  0.38985337,  1.31522366, -1.33654918,  0.35145247,
         0.24495861,  0.19222922, -0.09022656,  0.61649998,  0.37755096,
        -0.42229247, -0.76525802,  0.473443

## Example 2: DRlm - Classification

### Data Generating Process

In [8]:
def softmax(x):
    """
    Input: dim:n*(C-1)
    Output: softmax probabilities, dim:n*C
    
    """
    
    x_max = np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

np.random.seed(123)
n = 100; p = 5; L = 2; N = 1000
K = 2 # Number of classes
Xlist = [np.random.normal(0, 1, (n, p)) for _ in range(L)]
X0 = np.random.normal(0.1, 1, (N, p))
beta_list = [np.column_stack((np.zeros(p),np.random.normal(0, 0.25, (p,K)))) for _ in range(L)]
logits_list = [
    X.dot(beta) - np.mean(X.dot(beta))
    for X, beta in zip(Xlist, beta_list)
]
probs_list = [softmax(logits) for logits in logits_list]
ylist = [np.array([np.random.multinomial(1, probs[i, :]).tolist().index(1) for i in range(n)]) for probs in probs_list]


### Implementation

In [9]:
cc = DRlm.Classification(f_learner='linear', w_learner='linear')
cc.fit(Xlist,ylist,X0)
cc.infer()

## time cost: 4.8s

In [10]:
cc.probaX0_list

[array([[0.29682613, 0.12979242, 0.57338145],
        [0.40501283, 0.09072757, 0.5042596 ],
        [0.19407577, 0.5199716 , 0.28595263],
        ...,
        [0.31246474, 0.29167619, 0.39585907],
        [0.1130981 , 0.56827863, 0.31862327],
        [0.27045044, 0.20072325, 0.52882631]]),
 array([[0.21257647, 0.27065699, 0.51676653],
        [0.4909518 , 0.07680269, 0.43224551],
        [0.14379986, 0.69056886, 0.16563128],
        ...,
        [0.13593234, 0.60444461, 0.25962305],
        [0.12192151, 0.7299535 , 0.14812499],
        [0.20727164, 0.2734976 , 0.51923076]])]

### Results

In [11]:
cc.summary()

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.3583   0.6417

Fitted Coefficients:

Class 2 coefficients:
index     |        1        2        3        4        5
coef_     |   0.2330   0.0147  -0.1428   0.2283   0.7163

Class 3 coefficients:
index     |        1        2        3        4        5
coef_     |  -0.2345   0.6903   0.2463  -0.2958   0.2633

Confidence Intervals for each coefficient:

Class 2 Confidence Intervals:
index     |              1              2              3              4              5
CIs       | (-1.233,3.995) (-3.886,8.197) (-1.905,1.496) (-1.508,4.038) (-0.243,7.694)

Class 3 Confidence Intervals:
index     |              1              2              3              4              5
CIs       | (-1.764,1.550) (-1.791,13.050) (-0.877,3.586) (-2.148,1.260) (-2.278,6.150)



In [12]:
cc.summary(
    dim_search = [3,5], class_search=3
)

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.3583   0.6417

Fitted Coefficients:

Class 3 coefficients:
index     |        3        5
coef_     |   0.2463   0.2633

Confidence Intervals for each coefficient:

Class 3 Confidence Intervals:
index     |              3              5
CIs       | (-0.877,3.586) (-2.278,6.150)



In [13]:
cc.predict_proba()

{'pred_proba': array([[0.18597781, 0.17658149, 0.63744071],
        [0.44065281, 0.12374689, 0.43560029],
        [0.25250153, 0.46645683, 0.28104164],
        ...,
        [0.18052818, 0.28350621, 0.53596561],
        [0.19853889, 0.60943075, 0.19203037],
        [0.12645667, 0.12713534, 0.74640799]])}

In [14]:
cc.predict()

{'pred': array([2, 0, 1, 2, 2, 2, 2, 1, 0, 1, 1, 2, 1, 1, 0, 1, 0, 1, 2, 2, 2, 1,
        2, 2, 1, 2, 0, 2, 1, 0, 2, 2, 0, 2, 0, 1, 0, 1, 2, 0, 0, 2, 1, 1,
        2, 2, 1, 1, 1, 1, 2, 2, 1, 2, 1, 2, 0, 0, 1, 2, 2, 2, 1, 2, 2, 2,
        0, 1, 2, 2, 0, 1, 1, 2, 1, 1, 1, 0, 2, 1, 1, 0, 2, 1, 2, 2, 1, 0,
        2, 2, 1, 2, 2, 1, 1, 1, 2, 2, 0, 1, 2, 0, 0, 1, 2, 0, 2, 1, 0, 0,
        1, 1, 0, 0, 2, 2, 2, 1, 1, 2, 2, 1, 1, 1, 0, 0, 2, 2, 2, 2, 1, 2,
        0, 2, 2, 1, 1, 1, 0, 2, 0, 2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 0, 1, 2,
        2, 1, 1, 2, 1, 0, 1, 2, 2, 1, 0, 2, 2, 1, 0, 1, 0, 2, 1, 1, 0, 2,
        2, 2, 2, 1, 1, 0, 1, 2, 1, 0, 0, 2, 0, 0, 2, 2, 0, 1, 2, 2, 2, 1,
        1, 0, 0, 2, 2, 1, 1, 1, 1, 2, 1, 1, 1, 0, 2, 1, 1, 2, 0, 2, 0, 1,
        1, 0, 2, 1, 2, 1, 2, 1, 2, 2, 2, 1, 2, 1, 0, 1, 2, 2, 1, 1, 1, 2,
        0, 2, 0, 2, 1, 1, 0, 2, 1, 1, 0, 1, 1, 2, 1, 1, 1, 2, 2, 2, 0, 0,
        0, 1, 1, 1, 1, 1, 0, 2, 0, 2, 1, 2, 2, 0, 2, 2, 2, 1, 0, 2, 1, 1,
        1, 0, 2, 2, 2, 2, 0, 1

## Example 3: DRoL

### Data Generating Process

In [15]:
from data import *
data = DataContainerSimu1(n=2000, N=20000)
data.generate_funcs_list(L=2, seed=0)
data.generate_data()

Xlist = data.X_sources_list
Ylist = data.Y_sources_list
X0 = data.X_target

### Implementation

In [16]:
drol = DRoL(f_learner='xgb', w_learner='linear')
drol.fit(Xlist,Ylist,X0)

## time cost: 8.7s

### Results

In [17]:
drol.weight_

array([0.22519241, 0.77480759])

In [18]:
drol.predict()

{'pred': array([-0.97765349,  0.40863161,  0.41007573, ...,  1.87703664,
         0.56752075, -0.08127585])}